In [0]:
# ============================================================
# SILVER LAYER - UPDATED FOR CI/CD (USES GITHUB SECRETS)
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    raw_container = 'raw-dev'
    bronze_container = 'bronze-dev'
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    raw_container = 'raw-test'
    bronze_container = 'bronze-test'
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    raw_container = 'raw'
    bronze_container = 'bronze'
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
BRONZE = f"abfss://{bronze_container}@{storage_account_name}.dfs.core.windows.net"
RAW    = f"abfss://{raw_container}@{storage_account_name}.dfs.core.windows.net"
SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD   = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 BRONZE Container: {bronze_container}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📍 BRONZE Path: {BRONZE}")
print(f"📍 SILVER Path: {SILVER}")
print(f"{'='*55}\n")

print("✅ Config ready!")



✅ Config ready!


In [0]:
df_txn = spark.read.format("delta").load(f"{BRONZE}/transactions")

print(f"📊 Bronze Transactions loaded: {df_txn.count():,} rows")
df_txn.printSchema()

📊 Bronze Transactions loaded: 986,864 rows
root
 |-- transaction_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_type: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- outstanding_balance: double (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

df_txn = df_txn\
    .withColumn("transaction_id",      F.col("transaction_id").cast("long"))\
    .withColumn("account_id",          F.col("account_id").cast("long"))\
    .withColumn("customer_id",         F.col("customer_id").cast("long"))\
    .withColumn("amount",              F.col("amount").cast("double"))\
    .withColumn("outstanding_balance", F.col("outstanding_balance").cast("double"))\
    .withColumn("transaction_date",    F.to_date("transaction_date"))\
    .withColumn("ingestion_timestamp", F.to_timestamp("ingestion_timestamp"))

print("✅ Data types fixed!")
df_txn.printSchema()

✅ Data types fixed!
root
 |-- transaction_id: long (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- account_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_type: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- outstanding_balance: double (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



Remove Duplicates (Deterministic Deduplication)

In [0]:
from pyspark.sql.window import Window

window_dedup = Window.partitionBy("transaction_id").orderBy(
    F.col("ingestion_timestamp").desc()
)

df_deduped = df_txn.withColumn(
    "row_num",
    F.row_number().over(window_dedup)
).filter(
    F.col("row_num") == 1
).drop("row_num")

removed = df_txn.count() - df_deduped.count()
print(f"✅ Duplicates removed : {removed:,}")
print(f"✅ Rows after dedup   : {df_deduped.count():,}")

✅ Duplicates removed : 12,561
✅ Rows after dedup   : 974,303


Fix Negative Values

In [0]:
# Flag and fix negative amounts and balances
df_cleaned = df_deduped\
    .withColumn("amount",
        F.when(F.col("amount") < 0, F.abs(F.col("amount")))
         .otherwise(F.col("amount")))\
    .withColumn("outstanding_balance",
        F.when(F.col("outstanding_balance") < 0, F.lit(0.0))
         .otherwise(F.col("outstanding_balance")))\
    .withColumn("data_quality_flag",
        F.when(F.col("amount") < 0, F.lit("negative_amount_fixed"))
         .when(F.col("outstanding_balance") < 0, F.lit("negative_balance_fixed"))
         .otherwise(F.lit("clean")))

print("✅ Negative values fixed!")
df_cleaned.groupBy("data_quality_flag").count().show()

✅ Negative values fixed!
+-----------------+------+
|data_quality_flag| count|
+-----------------+------+
|            clean|974303|
+-----------------+------+



In [0]:
df_cleaned = df_cleaned\
    .withColumn("transaction_type",
        F.initcap(F.trim(F.col("transaction_type"))))\
    .withColumn("product_type",
        F.initcap(F.trim(F.col("product_type"))))\
    .withColumn("transaction_status",
        F.initcap(F.trim(F.col("transaction_status"))))

print("✅ transaction_type standardized:")
df_cleaned.groupBy("transaction_type").count().orderBy("transaction_type").show()

✅ transaction_type standardized:
+----------------+------+
|transaction_type| count|
+----------------+------+
|    Disbursement|243307|
|             Fee|243923|
|        Interest|243201|
|       Repayment|243872|
+----------------+------+



In [0]:
from pyspark.sql import functions as F

print("=" * 60)
print("  BRONZE TRANSACTIONS — DATA QUALITY REPORT")
print("=" * 60)

total = df_txn.count()
print(f"\n  📊 TOTAL RAW ROWS          : {total:>10,}")

# --- DUPLICATES ---
dupes = total - df_deduped.count()
print(f"\n  🔁 DUPLICATES")
print(f"     Duplicate rows removed   : {dupes:>10,}")
print(f"     Rows after dedup         : {df_deduped.count():>10,}")

# --- NULLS per column ---
print(f"\n  ❌ NULL VALUES PER COLUMN")
null_counts = df_txn.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_txn.columns
]).collect()[0].asDict()

for col_name, null_count in null_counts.items():
    if null_count > 0:
        pct = (null_count / total) * 100
        print(f"     {col_name:<30} {null_count:>8,}  ({pct:.2f}%)")
    else:
        print(f"     {col_name:<30} {null_count:>8,}  ✅ clean")

# --- NEGATIVE VALUES ---
print(f"\n  ⚠️  NEGATIVE VALUES")
neg_amount = df_txn.filter(F.col("amount").cast("double") < 0).count()
neg_balance = df_txn.filter(F.col("outstanding_balance").cast("double") < 0).count()
print(f"     Negative amount          : {neg_amount:>10,}")
print(f"     Negative outstanding_bal : {neg_balance:>10,}")

# --- TRANSACTION STATUS BREAKDOWN ---
print(f"\n  📋 TRANSACTION STATUS BREAKDOWN")
df_txn.groupBy("transaction_status")\
    .count()\
    .orderBy("transaction_status")\
    .collect()
for row in df_txn.groupBy("transaction_status").count().orderBy("transaction_status").collect():
    print(f"     {row['transaction_status']:<20} {row['count']:>10,}")

# --- PRODUCT TYPE BREAKDOWN ---
print(f"\n  📋 PRODUCT TYPE BREAKDOWN")
for row in df_txn.groupBy("product_type").count().orderBy("product_type").collect():
    print(f"     {row['product_type']:<20} {row['count']:>10,}")

# --- TRANSACTION TYPE BREAKDOWN ---
print(f"\n  📋 TRANSACTION TYPE BREAKDOWN")
for row in df_txn.groupBy("transaction_type").count().orderBy("transaction_type").collect():
    print(f"     {row['transaction_type']:<20} {row['count']:>10,}")

# --- DATE RANGE ---
print(f"\n  📅 DATE RANGE")
date_range = df_txn.agg(
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date")
).collect()[0]
print(f"     Earliest transaction     : {date_range['min_date']}")
print(f"     Latest transaction       : {date_range['max_date']}")

print("\n" + "=" * 60)
print("  ✅ PROFILING COMPLETE!")
print("=" * 60)

  BRONZE TRANSACTIONS — DATA QUALITY REPORT

  📊 TOTAL RAW ROWS          :    986,864

  🔁 DUPLICATES
     Duplicate rows removed   :     12,561
     Rows after dedup         :    974,303

  ❌ NULL VALUES PER COLUMN
     transaction_id                        0  ✅ clean
     transaction_date                      0  ✅ clean
     account_id                            0  ✅ clean
     customer_id                           0  ✅ clean
     product_type                          0  ✅ clean
     transaction_type                      0  ✅ clean
     amount                                0  ✅ clean
     outstanding_balance                   0  ✅ clean
     transaction_status                    0  ✅ clean
     ingestion_timestamp                   0  ✅ clean
     source_file                           0  ✅ clean
     year                                  0  ✅ clean
     month                                 0  ✅ clean

  ⚠️  NEGATIVE VALUES
     Negative amount          :         50
     Negative ou

In [0]:
print("=" * 60)
print("  MASTER TABLES — DATA QUALITY REPORT")
print("=" * 60)

# ── Load all master tables from Bronze ──
df_cust  = spark.read.format("delta").load(f"{BRONZE}/customer_master")
df_acc   = spark.read.format("delta").load(f"{BRONZE}/account_master")
df_br    = spark.read.format("delta").load(f"{BRONZE}/branch_channel")
df_del   = spark.read.format("delta").load(f"{BRONZE}/repayment_delinquency")

# ============================================================
# HELPER FUNCTION
# ============================================================
def profile_table(df, table_name):
    total = df.count()
    print(f"\n{'=' * 60}")
    print(f"  📁 TABLE : {table_name}")
    print(f"{'=' * 60}")
    print(f"  📊 TOTAL ROWS               : {total:>10,}")
    print(f"  📊 TOTAL COLUMNS            : {len(df.columns):>10,}")

    # --- NULLS ---
    print(f"\n  ❌ NULL VALUES PER COLUMN")
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    for col_name, null_count in null_counts.items():
        pct = (null_count / total) * 100
        status = f"({pct:.2f}%)" if null_count > 0 else "✅ clean"
        print(f"     {col_name:<35} {null_count:>8,}  {status}")

    # --- DUPLICATES ---
    dup_count = total - df.dropDuplicates().count()
    print(f"\n  🔁 DUPLICATE ROWS           : {dup_count:>10,}")

    return total

# ============================================================
# 1. CUSTOMER MASTER
# ============================================================
profile_table(df_cust, "customer_master")

print(f"\n  📋 GENDER BREAKDOWN")
for row in df_cust.groupBy("gender").count().orderBy("gender").collect():
    print(f"     {row['gender']:<25} {row['count']:>10,}")

print(f"\n  📋 EMPLOYMENT TYPE BREAKDOWN")
for row in df_cust.groupBy("employment_type").count().orderBy("employment_type").collect():
    print(f"     {row['employment_type']:<25} {row['count']:>10,}")

print(f"\n  📋 RISK SEGMENT BREAKDOWN")
for row in df_cust.groupBy("risk_segment").count().orderBy("risk_segment").collect():
    print(f"     {row['risk_segment']:<25} {row['count']:>10,}")

print(f"\n  📋 CITY BREAKDOWN")
for row in df_cust.groupBy("city").count().orderBy("city").collect():
    print(f"     {row['city']:<25} {row['count']:>10,}")

print(f"\n  📊 ANNUAL INCOME STATS")
df_cust.select(
    F.round(F.min("annual_income"),2).alias("min"),
    F.round(F.max("annual_income"),2).alias("max"),
    F.round(F.avg("annual_income"),2).alias("avg"),
    F.count(F.when(F.col("annual_income").isNull(), 1)).alias("nulls")
).show()

print(f"\n  📊 AGE STATS")
df_cust.select(
    F.min("age").alias("min_age"),
    F.max("age").alias("max_age"),
    F.round(F.avg("age"),1).alias("avg_age")
).show()

# ============================================================
# 2. ACCOUNT MASTER
# ============================================================
profile_table(df_acc, "account_master")

print(f"\n  📋 PRODUCT TYPE BREAKDOWN")
for row in df_acc.groupBy("product_type").count().orderBy("product_type").collect():
    print(f"     {row['product_type']:<25} {row['count']:>10,}")

print(f"\n  📋 ACCOUNT STATUS BREAKDOWN")
for row in df_acc.groupBy("account_status").count().orderBy("account_status").collect():
    print(f"     {row['account_status']:<25} {row['count']:>10,}")

print(f"\n  📊 CREDIT LIMIT STATS")
df_acc.select(
    F.min("credit_limit").alias("min"),
    F.max("credit_limit").alias("max"),
    F.round(F.avg("credit_limit"),2).alias("avg"),
    F.count(F.when(F.col("credit_limit") == 0, 1)).alias("zero_credit_limit")
).show()

print(f"\n  📊 INTEREST RATE STATS")
df_acc.select(
    F.round(F.min("interest_rate"),2).alias("min_rate"),
    F.round(F.max("interest_rate"),2).alias("max_rate"),
    F.round(F.avg("interest_rate"),2).alias("avg_rate")
).show()

# ============================================================
# 3. BRANCH CHANNEL
# ============================================================
profile_table(df_br, "branch_channel")

print(f"\n  📋 CHANNEL TYPE BREAKDOWN")
for row in df_br.groupBy("channel_type").count().orderBy("channel_type").collect():
    print(f"     {row['channel_type']:<25} {row['count']:>10,}")

print(f"\n  📋 REGION BREAKDOWN")
for row in df_br.groupBy("region").count().orderBy("region").collect():
    print(f"     {row['region']:<25} {row['count']:>10,}")

# ============================================================
# 4. REPAYMENT DELINQUENCY
# ============================================================
profile_table(df_del, "repayment_delinquency")

print(f"\n  📋 DELINQUENCY FLAG BREAKDOWN")
for row in df_del.groupBy("delinquency_flag").count().orderBy("delinquency_flag").collect():
    flag = "⚠️ Delinquent" if row['delinquency_flag'] == 1 else "✅ On Time"
    print(f"     {flag:<25} {row['count']:>10,}")

print(f"\n  📊 DAYS PAST DUE STATS")
df_del.select(
    F.min("days_past_due").alias("min_dpd"),
    F.max("days_past_due").alias("max_dpd"),
    F.round(F.avg("days_past_due"),2).alias("avg_dpd"),
    F.count(F.when(F.col("days_past_due") < 0, 1)).alias("negative_dpd")
).show()

print(f"\n  📊 OVERDUE AMOUNT STATS")
df_del.select(
    F.round(F.min("overdue_amount"),2).alias("min"),
    F.round(F.max("overdue_amount"),2).alias("max"),
    F.round(F.avg("overdue_amount"),2).alias("avg"),
    F.round(F.sum("overdue_amount"),2).alias("total")
).show()

print(f"\n  📋 DAYS PAST DUE BUCKETS")
df_del.withColumn("dpd_bucket",
    F.when(F.col("days_past_due") == 0,  F.lit("0 - On Time"))
     .when(F.col("days_past_due") <= 10, F.lit("1-10 days"))
     .when(F.col("days_past_due") <= 30, F.lit("11-30 days"))
     .when(F.col("days_past_due") <= 60, F.lit("31-60 days"))
     .otherwise(F.lit("60+ days"))
).groupBy("dpd_bucket").count().orderBy("dpd_bucket").show()

print("=" * 60)
print("  ✅ ALL MASTER TABLES PROFILING COMPLETE!")
print("=" * 60)

  MASTER TABLES — DATA QUALITY REPORT

  📁 TABLE : customer_master
  📊 TOTAL ROWS               :    100,000
  📊 TOTAL COLUMNS            :          9

  ❌ NULL VALUES PER COLUMN
     customer_id                                0  ✅ clean
     age                                        0  ✅ clean
     gender                                     0  ✅ clean
     city                                       0  ✅ clean
     employment_type                            0  ✅ clean
     annual_income                            499  (0.50%)
     risk_segment                               0  ✅ clean
     ingestion_timestamp                        0  ✅ clean
     source_file                                0  ✅ clean

  🔁 DUPLICATE ROWS           :          0

  📋 GENDER BREAKDOWN
     Female                        49,880
     Male                          50,120

  📋 EMPLOYMENT TYPE BREAKDOWN
     Business                      20,144
     Retired                       19,996
     Salaried             

In [0]:
# Derive: exposure, interest_accrued, risk_band
df_enriched = df_cleaned\
    .withColumn("exposure",
        F.round(F.col("outstanding_balance") * F.lit(1.1), 2))\
    .withColumn("interest_accrued",
        F.when(F.col("transaction_type") == "Interest", F.col("amount"))
         .otherwise(F.lit(0.0)))\
    .withColumn("risk_band",
        F.when(F.col("outstanding_balance") > 800000,  F.lit("High"))
         .when(F.col("outstanding_balance") > 400000,  F.lit("Medium"))
         .otherwise(F.lit("Low")))\
    .withColumn("silver_processed_at", F.current_timestamp())

print("✅ Derived fields added!")
df_enriched.select(
    "transaction_id","outstanding_balance",
    "exposure","interest_accrued","risk_band"
).show(5)

✅ Derived fields added!
+--------------+-------------------+---------+----------------+---------+
|transaction_id|outstanding_balance| exposure|interest_accrued|risk_band|
+--------------+-------------------+---------+----------------+---------+
|       9000000|          629071.19|691978.31|             0.0|   Medium|
|       9000007|          276281.64| 303909.8|        29350.95|      Low|
|       9000009|          713878.47|785266.32|        15276.21|   Medium|
|       9000011|          225012.52|247513.77|             0.0|      Low|
|       9000015|           21812.66| 23993.93|             0.0|      Low|
+--------------+-------------------+---------+----------------+---------+
only showing top 5 rows


In [0]:
# Customer Master — fill missing income
df_customer = spark.read.format("delta").load(f"{BRONZE}/customer_master")
median_income = df_customer.approxQuantile("annual_income", [0.5], 0.01)[0]

df_customer_silver = df_customer\
    .withColumn("annual_income",
        F.when(F.col("annual_income").isNull(), F.lit(median_income))
         .otherwise(F.col("annual_income")))\
    .withColumn("risk_segment",
        F.when(F.col("risk_segment").isNull(), F.lit("Medium"))
         .otherwise(F.col("risk_segment")))

# Account Master — fix zero credit limits
df_account = spark.read.format("delta").load(f"{BRONZE}/account_master")
df_account_silver = df_account\
    .withColumn("credit_limit",
        F.when(F.col("credit_limit") == 0, F.lit(50000))
         .otherwise(F.col("credit_limit")))

# Branch & Delinquency — fix negative days_past_due
df_branch  = spark.read.format("delta").load(f"{BRONZE}/branch_channel")
df_deliq   = spark.read.format("delta").load(f"{BRONZE}/repayment_delinquency")

df_deliq_silver = df_deliq\
    .withColumn("days_past_due",
        F.when(F.col("days_past_due") < 0, F.lit(0))
         .otherwise(F.col("days_past_due")))\
    .withColumn("delinquency_flag",
        F.when(F.col("days_past_due") > 0, F.lit(1))
         .otherwise(F.lit(0)))

print("✅ Master tables cleaned!")

✅ Master tables cleaned!


In [0]:
# Transactions Silver
df_enriched.write\
    .format("delta")\
    .mode("overwrite")\
    .partitionBy("product_type", "year")\
    .save(f"{silver}/transactions")

# Master tables Silver
df_customer_silver.write.format("delta").mode("overwrite").save(f"{silver}/customer_master")
df_account_silver.write.format("delta").mode("overwrite").save(f"{silver}/account_master")
df_branch.write.format("delta").mode("overwrite").save(f"{silver}/branch_channel")
df_deliq_silver.write.format("delta").mode("overwrite").save(f"{silver}/repayment_delinquency")

print("✅ All Silver tables written!")

✅ All Silver tables written!
